# Упражнение: PPO: преимущества GAE и clipped-цель по шагам роллаута

## Задание

В `input.csv` — роллаут из 16 шагов: `t, reward, done, value, next_value, logp_old, logp_new`.
`value` = V(sₜ) критика, `next_value` = V(sₜ₊₁) — оценка состояния после шага (на шаге с `done = 1`
это уже стартовое состояние нового эпизода); `logp_old` — log π_old(aₜ | sₜ) при сборе данных,
`logp_new` — log π_θ(aₜ | sₜ) текущей политики после нескольких шагов оптимизации.
Последний шаг (t = 15) — не конец эпизода: роллаут просто оборван, ценность после него бутстрапится.

Параметры: γ = 0.99, λ = 0.95, ε = 0.2.

1. **TD-ошибка:** δₜ = rₜ + γ·(1 − doneₜ)·V(sₜ₊₁) − V(sₜ).
2. **GAE** (с конца): Aₜ = δₜ + γλ·(1 − doneₜ)·Aₜ₊₁, A после последнего шага = 0.
3. **Возврат для критика:** retₜ = Aₜ + V(sₜ).
4. **Нормализация преимуществ:** A_norm = (A − mean(A)) / (std(A) + 1e-8), std **несмещённое**
   (`ddof=1`, как `torch.std`).
5. **Отношение вероятностей:** ratio = exp(logp_new − logp_old).
6. **Clipped-цель на шаге:** L = min(ratio·A_norm, clip(ratio, 1 − ε, 1 + ε)·A_norm).
7. **Флаг обрезки** `clipped` = 1, если (A_norm > 0 и ratio > 1 + ε) или (A_norm < 0 и ratio < 1 − ε) —
   на таком шаге градиент цели по θ равен нулю; иначе 0.

## Подсказки

- Граница эпизода: на шаге с `done = 1` нельзя ни бутстрапить V(sₜ₊₁), ни тянуть Aₜ₊₁ из следующего эпизода —
  обе связи обрываются множителем (1 − done).
- Обрезается только **выгода**: при A_norm < 0 и ratio > 1 + ε минимум выберет необрезанное
  ratio·A_norm (штраф полностью учитывается), и `clipped` = 0.
- Цикл `for t in reversed(range(16))`: `delta = r[t] + g*(1-d[t])*nv[t] - v[t]`,
  `last = delta + g*lam*(1-d[t])*last`; дальше всё векторно: `np.minimum(ratio*A, np.clip(ratio, 0.8, 1.2)*A)`.

## Ожидаемый формат выходного файла

CSV с колонками `t, delta, adv, ret, adv_norm, ratio, l_clip, clipped` — 16 строк (t = 0..15 по порядку);
`clipped` — 0 или 1; остальные значения округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/ppo_reinforcement_learning
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.